In [1]:
!pip install ipykernel


In [135]:
import os
import psycopg2
from psycopg2 import sql

# Optionally, load environment variables from a .env file
# from dotenv import load_dotenv
# load_dotenv()

# Retrieve database connection parameters from environment variables
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

def get_connection():
    """
    Establishes and returns a new database connection.
    """
    conn = psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    return conn


def fetch_rows(query, params=None):
    """
    Executes a SELECT query and returns all rows.

    :param query: SQL query string with placeholders.
    :param params: Tuple of parameters to pass into query.
    :return: List of result rows.
    """
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(query, params)
            rows = cur.fetchall()
            return rows
    finally:
        conn.close()


def execute_query(query, params=None):
    """
    Executes an INSERT/UPDATE/DELETE query and commits the transaction.

    :param query: SQL query string with placeholders.
    :param params: Tuple of parameters to pass into query.
    :return: Number of affected rows.
    """
    conn = get_connection()
    try:
        with conn.cursor() as cur:
            cur.execute(query, params)
            conn.commit()
            return cur.rowcount
    except Exception as e:
        conn.rollback()
        raise
    finally:
        conn.close()


# Example usage
if __name__ == '__main__':
    # Fetch users older than 30
    select_query = """SELECT DEVICE_CODE, COUNT(DEVICE_CODE) AS usage_count FROM devices GROUP BY DEVICE_CODE ORDER BY usage_count DESC LIMIT 5;"""

    users = fetch_rows(select_query)

   
users

UndefinedColumn: column "device_code" does not exist
LINE 1: SELECT DEVICE_CODE, COUNT(DEVICE_CODE) AS usage_count FROM d...
               ^


In [3]:
!pip install psycopg2-binary


  Using cached psycopg2_binary-2.9.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)


In [4]:
!pip install azure-core


  Using cached azure_core-1.34.0-py3-none-any.whl (207 kB)


In [6]:
import os
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential

endpoint = "https://azureopenaiins-01.openai.azure.com/"
model_name = "text-embedding-3-large"
deployment = "TE3-Large-ins2"

api_version = "2024-02-01"

client = AzureOpenAI(
api_version="2024-12-01-preview",
azure_endpoint=endpoint,
credential=AzureKeyCredential("6v7UWVpBP2tKOaM8Q5RiLJUayH0SiCt7kc4fh28CC7DMHNP46TYVJQQJ99BDACYeBjFXJ3w3AAABACOGyWQ1")
)

response = client.embeddings.create(
input=["first phrase","second phrase","third phrase"],
model=deployment
)

for item in response.data:
    length = len(item.embedding)
    print(
    f"data[{item.index}]: length={length}, "
    f"[{item.embedding[0]}, {item.embedding[1]}, "
    f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)

TypeError: AzureOpenAI.__init__() got an unexpected keyword argument 'credential'

In [7]:
from openai import AzureOpenAI

endpoint = "https://azureopenaiins-01.openai.azure.com/"
deployment = "TE3-Large-ins2"
api_key = "6v7UWVpBP2tKOaM8Q5RiLJUayH0SiCt7kc4fh28CC7DMHNP46TYVJQQJ99BDACYeBjFXJ3w3AAABACOGyWQ1"

client = AzureOpenAI(
    api_key=api_key,
    api_version="2024-02-01",
    azure_endpoint=endpoint,
)

response = client.embeddings.create(
    input=["first phrase", "second phrase", "third phrase"],
    model=deployment,
)

for item in response.data:
    length = len(item.embedding)
    print(
        f"data[{item.index}]: length={length}, "
        f"[{item.embedding[0]}, {item.embedding[1]}, "
        f"..., {item.embedding[length-2]}, {item.embedding[length-1]}]"
    )
print(response.usage)


data[0]: length=3072, [0.022330209612846375, -0.002088305074721575, ..., -0.014379994943737984, 0.006100048776715994]
data[1]: length=3072, [0.011640272103250027, 0.005252661183476448, ..., -0.028720801696181297, -0.0025770869106054306]
data[2]: length=3072, [0.016326788812875748, -0.0018455119570717216, ..., -0.005349587649106979, 0.006049444433301687]
Usage(prompt_tokens=6, total_tokens=6)


Connecting to PostgreSQL database...
Database error: connection to server at "localhost" (127.0.0.1), port 5432 failed: FATAL:  password authentication failed for user "your_username"
connection to server at "localhost" (127.0.0.1), port 5432 failed: FATAL:  password authentication failed for user "your_username"

Failed to extract schema data.


In [39]:
import psycopg2
import json
import os
from typing import Dict, List, Any
from dotenv import load_dotenv

def get_database_schema(host: str, database: str, user: str, password: str, port: int = 5432) -> Dict[str, Any]:
    """
    Extract database schema information from PostgreSQL database.
    
    Args:
        host: Database host
        database: Database name
        user: Database user
        password: Database password
        port: Database port (default: 5432)
    
    Returns:
        Dictionary containing database schema information
    """
    try:
        # Connect to PostgreSQL database
        conn = psycopg2.connect(
            host=host,
            database=database,
            user=user,
            password=password,
            port=port
        )
        
        cursor = conn.cursor()
        
        # Query to get all tables, columns, and their data types
        query = """
        SELECT 
            t.table_schema,
            t.table_name,
            c.column_name,
            c.data_type,
            c.is_nullable,
            c.column_default,
            c.character_maximum_length,
            c.numeric_precision,
            c.numeric_scale,
            c.ordinal_position
        FROM 
            information_schema.tables t
        JOIN 
            information_schema.columns c ON t.table_name = c.table_name 
            AND t.table_schema = c.table_schema
        WHERE 
            t.table_type = 'BASE TABLE'
            AND t.table_schema NOT IN ('information_schema', 'pg_catalog')
        ORDER BY 
            t.table_schema, t.table_name, c.ordinal_position;
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        # Structure the data
        schema_info = {
            "database_name": database,
            "extraction_timestamp": None,  # You can add datetime.now().isoformat() if needed
            "schemas": {}
        }
        
        for row in results:
            schema_name = row[0]
            table_name = row[1]
            column_name = row[2]
            data_type = row[3]
            is_nullable = row[4]
            column_default = row[5]
            char_max_length = row[6]
            numeric_precision = row[7]
            numeric_scale = row[8]
            ordinal_position = row[9]
            
            # Initialize schema if not exists
            if schema_name not in schema_info["schemas"]:
                schema_info["schemas"][schema_name] = {"tables": {}}
            
            # Initialize table if not exists
            if table_name not in schema_info["schemas"][schema_name]["tables"]:
                schema_info["schemas"][schema_name]["tables"][table_name] = {"columns": []}
            
            # Add column information
            column_info = {
                "column_name": column_name,
                "data_type": data_type,
                "is_nullable": is_nullable,
                "column_default": column_default,
                "character_maximum_length": char_max_length,
                "numeric_precision": numeric_precision,
                "numeric_scale": numeric_scale,
                "ordinal_position": ordinal_position
            }
            
            schema_info["schemas"][schema_name]["tables"][table_name]["columns"].append(column_info)
        
        cursor.close()
        conn.close()
        
        return schema_info
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None

def save_schema_to_json(schema_data: Dict[str, Any], filename: str = "database_schema.json") -> bool:
    """
    Save schema data to JSON file.
    
    Args:
        schema_data: Schema data dictionary
        filename: Output filename
    
    Returns:
        True if successful, False otherwise
    """
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(schema_data, f, indent=2, ensure_ascii=False, default=str)
        print(f"Schema saved to {filename}")
        return True
    except Exception as e:
        print(f"Error saving to JSON: {e}")
        return False

def load_db_config():
    """
    Load database configuration from .env file.
    
    Returns:
        Dictionary containing database configuration
    """
    # Load environment variables from .env file
    load_dotenv()
    
    # Get database configuration from environment variables
    db_config = {
        "host": os.getenv("DB_HOST", "localhost"),
        "database": os.getenv("DB_NAME"),
        "user": os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
        "port": int(os.getenv("DB_PORT", 5432))
    }
    
    # Validate required environment variables
    required_vars = ["DB_NAME", "DB_USER", "DB_PASSWORD"]
    missing_vars = [var for var in required_vars if not os.getenv(var)]
    
    if missing_vars:
        raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")
    
    return db_config

def main():
    """
    Main function to extract and save database schema.
    """
    try:
        # Load database configuration from .env file
        db_config = load_db_config()
        print(f"Connecting to database: {db_config['database']} at {db_config['host']}:{db_config['port']}")
    
    except ValueError as e:
        print(f"Configuration error: {e}")
        print("Please ensure your .env file contains:")
        print("DB_HOST=localhost")
        print("DB_PORT=5432")
        print("DB_NAME=your_database")
        print("DB_USER=your_username")
        print("DB_PASSWORD=your_password")
        return
    except Exception as e:
        print(f"Error loading configuration: {e}")
        return
    
    print("Connecting to PostgreSQL database...")
    schema_data = get_database_schema(**db_config)
    
    if schema_data:
        print("Schema extraction successful!")
        
        # Save to JSON file
        save_schema_to_json(schema_data)
        
        # Print summary
        total_tables = sum(len(schema["tables"]) for schema in schema_data["schemas"].values())
        total_columns = sum(
            len(table["columns"]) 
            for schema in schema_data["schemas"].values() 
            for table in schema["tables"].values()
        )
        
        print(f"\nSummary:")
        print(f"- Schemas: {len(schema_data['schemas'])}")
        print(f"- Tables: {total_tables}")
        print(f"- Columns: {total_columns}")
        
        # Optional: Print first few tables as example
        print(f"\nFirst few tables:")
        count = 0
        for schema_name, schema in schema_data["schemas"].items():
            for table_name, table in schema["tables"].items():
                if count < 3:  # Show first 3 tables
                    print(f"- {schema_name}.{table_name} ({len(table['columns'])} columns)")
                    for col in table["columns"][:3]:  # Show first 3 columns
                        print(f"  - {col['column_name']}: {col['data_type']}")
                    if len(table["columns"]) > 3:
                        print(f"  - ... and {len(table['columns']) - 3} more columns")
                    count += 1
        
    else:
        print("Failed to extract schema data.")

# Alternative function for simplified output (just table and column names with types)
def get_simplified_schema_from_env() -> Dict[str, Any]:
    """
    Extract simplified database schema using .env configuration.
    """
    try:
        db_config = load_db_config()
        return get_simplified_schema(**db_config)
    except Exception as e:
        print(f"Error: {e}")
        return None

def get_simplified_schema(host: str, database: str, user: str, password: str, port: int = 5432) -> Dict[str, Any]:
    """
    Extract simplified database schema (table names, column names, and data types only).
    """
    try:
        conn = psycopg2.connect(
            host=host, database=database, user=user, password=password, port=port
        )
        cursor = conn.cursor()
        
        query = """
        SELECT 
            t.table_name,
            c.column_name,
            c.data_type
        FROM 
            information_schema.tables t
        JOIN 
            information_schema.columns c ON t.table_name = c.table_name
        WHERE 
            t.table_type = 'BASE TABLE'
            AND t.table_schema = 'public'
        ORDER BY 
            t.table_name, c.ordinal_position;
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        simplified_schema = {"tables": {}}
        
        for row in results:
            table_name = row[0]
            column_name = row[1]
            data_type = row[2]
            
            if table_name not in simplified_schema["tables"]:
                simplified_schema["tables"][table_name] = {}
            
            simplified_schema["tables"][table_name][column_name] = data_type
        
        cursor.close()
        conn.close()
        
        return simplified_schema
        
    except Exception as e:
        print(f"Error: {e}")
        return None

if __name__ == "__main__":
    main()

Connecting to database: sample at localhost:5432
Connecting to PostgreSQL database...
Schema extraction successful!
Schema saved to database_schema.json

Summary:
- Schemas: 1
- Tables: 15
- Columns: 158

First few tables:
- public.allergies (6 columns)
  - START: date
  - STOP: date
  - PATIENT: character varying
  - ... and 3 more columns
- public.careplans (9 columns)
  - ID: character varying
  - START: date
  - STOP: date
  - ... and 6 more columns
- public.conditions (6 columns)
  - START: date
  - STOP: date
  - PATIENT: character varying
  - ... and 3 more columns


In [63]:
import json
from typing import Dict, List, Any

def generate_complete_formatted_schema() -> List[Dict]:
    """
    Generate complete formatted schema from the extracted database JSON.
    
    Returns:
        List of formatted schema dictionaries
    """
    
    # Your JSON data from the document
    json_data = {
        "database_name": "sample",
        "extraction_timestamp": None,
        "schemas": {
            "public": {
                "tables": {
                    "allergies": {
                        "columns": [
                            {"column_name": "START", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 5},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 6}
                        ]
                    },
                    "careplans": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "START", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 3},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 6},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "REASONCODE", "data_type": "bigint", "ordinal_position": 8},
                            {"column_name": "REASONDESCRIPTION", "data_type": "character varying", "ordinal_position": 9}
                        ]
                    },
                    "conditions": {
                        "columns": [
                            {"column_name": "START", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 5},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 6}
                        ]
                    },
                    "devices": {
                        "columns": [
                            {"column_name": "START", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 5},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "UDI", "data_type": "character varying", "ordinal_position": 7}
                        ]
                    },
                    "encounters": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "START", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 3},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "ORGANIZATION", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "PROVIDER", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "PAYER", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "ENCOUNTERCLASS", "data_type": "character varying", "ordinal_position": 8},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 9},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 10},
                            {"column_name": "BASE_ENCOUNTER_COST", "data_type": "numeric", "ordinal_position": 11},
                            {"column_name": "TOTAL_CLAIM_COST", "data_type": "numeric", "ordinal_position": 12},
                            {"column_name": "PAYER_COVERAGE", "data_type": "numeric", "ordinal_position": 13},
                            {"column_name": "REASONCODE", "data_type": "bigint", "ordinal_position": 14},
                            {"column_name": "REASONDESCRIPTION", "data_type": "character varying", "ordinal_position": 15}
                        ]
                    },
                    "imaging_studies": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "DATE", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "BODYSITE_CODE", "data_type": "bigint", "ordinal_position": 5},
                            {"column_name": "BODYSITE_DESCRIPTION", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "MODALITY_CODE", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "MODALITY_DESCRIPTION", "data_type": "character varying", "ordinal_position": 8},
                            {"column_name": "SOP_CODE", "data_type": "character varying", "ordinal_position": 9},
                            {"column_name": "SOP_DESCRIPTION", "data_type": "character varying", "ordinal_position": 10}
                        ]
                    },
                    "immunizations": {
                        "columns": [
                            {"column_name": "DATE", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 2},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 4},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "BASE_COST", "data_type": "numeric", "ordinal_position": 6}
                        ]
                    },
                    "medications": {
                        "columns": [
                            {"column_name": "START", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "STOP", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "PAYER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 6},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "BASE_COST", "data_type": "numeric", "ordinal_position": 8},
                            {"column_name": "PAYER_COVERAGE", "data_type": "numeric", "ordinal_position": 9},
                            {"column_name": "DISPENSES", "data_type": "integer", "ordinal_position": 10},
                            {"column_name": "TOTALCOST", "data_type": "numeric", "ordinal_position": 11},
                            {"column_name": "REASONCODE", "data_type": "bigint", "ordinal_position": 12},
                            {"column_name": "REASONDESCRIPTION", "data_type": "character varying", "ordinal_position": 13}
                        ]
                    },
                    "observations": {
                        "columns": [
                            {"column_name": "DATE", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 2},
                            {"column_name": "ENCOUTER", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "CODE", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "VALUE", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "UNITS", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "TYPE", "data_type": "text", "ordinal_position": 8}
                        ]
                    },
                    "organizations": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "NAME", "data_type": "text", "ordinal_position": 2},
                            {"column_name": "ADDRESS", "data_type": "text", "ordinal_position": 3},
                            {"column_name": "CITY", "data_type": "text", "ordinal_position": 4},
                            {"column_name": "STATE", "data_type": "text", "ordinal_position": 5},
                            {"column_name": "ZIP", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "LAT", "data_type": "numeric", "ordinal_position": 7},
                            {"column_name": "LON", "data_type": "numeric", "ordinal_position": 8},
                            {"column_name": "PHONE", "data_type": "character varying", "ordinal_position": 9},
                            {"column_name": "REVENUE", "data_type": "numeric", "ordinal_position": 10}
                        ]
                    },
                    "patients": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "BIRTHDATE", "data_type": "date", "ordinal_position": 2},
                            {"column_name": "DEATHDATE", "data_type": "date", "ordinal_position": 3},
                            {"column_name": "SSN", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "DRIVERS", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "PASSPORT", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "PREFIX", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "FIRST", "data_type": "text", "ordinal_position": 8},
                            {"column_name": "LAST", "data_type": "text", "ordinal_position": 9},
                            {"column_name": "SUFFIX", "data_type": "character varying", "ordinal_position": 10},
                            {"column_name": "MAIDEN", "data_type": "character varying", "ordinal_position": 11},
                            {"column_name": "MARTIAL", "data_type": "character varying", "ordinal_position": 12},
                            {"column_name": "RACE", "data_type": "character varying", "ordinal_position": 13},
                            {"column_name": "ETHINICITY", "data_type": "character varying", "ordinal_position": 14},
                            {"column_name": "GENDER", "data_type": "character", "ordinal_position": 15},
                            {"column_name": "BIRTHPLACE", "data_type": "character varying", "ordinal_position": 16},
                            {"column_name": "ADDRESS", "data_type": "character varying", "ordinal_position": 17},
                            {"column_name": "CITY", "data_type": "character varying", "ordinal_position": 18},
                            {"column_name": "STATE", "data_type": "character varying", "ordinal_position": 19},
                            {"column_name": "COUNTY", "data_type": "character varying", "ordinal_position": 20},
                            {"column_name": "ZIP", "data_type": "character varying", "ordinal_position": 21},
                            {"column_name": "LAT", "data_type": "numeric", "ordinal_position": 22},
                            {"column_name": "LON", "data_type": "numeric", "ordinal_position": 23},
                            {"column_name": "HEALTHCARE_EXPENSES", "data_type": "numeric", "ordinal_position": 24},
                            {"column_name": "HEALTHCARE_COVERAGE", "data_type": "numeric", "ordinal_position": 25}
                        ]
                    },
                    "payer_transitions": {
                        "columns": [
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "START_YEAR", "data_type": "integer", "ordinal_position": 2},
                            {"column_name": "END_YEAR", "data_type": "integer", "ordinal_position": 3},
                            {"column_name": "PAYER", "data_type": "character varying", "ordinal_position": 4},
                            {"column_name": "OWNERSHIP", "data_type": "text", "ordinal_position": 5}
                        ]
                    },
                    "payers": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "NAME", "data_type": "text", "ordinal_position": 2},
                            {"column_name": "ADDRESS", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "CITY", "data_type": "text", "ordinal_position": 4},
                            {"column_name": "STATE_HEADQUARTERED", "data_type": "text", "ordinal_position": 5},
                            {"column_name": "ZIP", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "PHONE", "data_type": "character varying", "ordinal_position": 7},
                            {"column_name": "AMOUNT_COVERED", "data_type": "numeric", "ordinal_position": 8},
                            {"column_name": "AMOUNT_UNCOVERED", "data_type": "numeric", "ordinal_position": 9},
                            {"column_name": "REVENUE", "data_type": "numeric", "ordinal_position": 10},
                            {"column_name": "COVERED_ENCOUTERS", "data_type": "bigint", "ordinal_position": 11},
                            {"column_name": "UNCOVERED_ENCOUNTERS", "data_type": "bigint", "ordinal_position": 12},
                            {"column_name": "COVERED_MEDICATIONS", "data_type": "bigint", "ordinal_position": 13},
                            {"column_name": "UNCOVERED_MEDICATIONS", "data_type": "bigint", "ordinal_position": 14},
                            {"column_name": "COVERED_PROCEDURES", "data_type": "bigint", "ordinal_position": 15},
                            {"column_name": "UNCOVERED_PROCEDURES", "data_type": "bigint", "ordinal_position": 16},
                            {"column_name": "COVERED_IMMUNIZATIONS", "data_type": "bigint", "ordinal_position": 17},
                            {"column_name": "UNCOVERED_IMMUNIZATIONS", "data_type": "bigint", "ordinal_position": 18},
                            {"column_name": "UNIQUE_CUSTOMERS", "data_type": "bigint", "ordinal_position": 19}
                        ]
                    },
                    "procedures": {
                        "columns": [
                            {"column_name": "DATE", "data_type": "date", "ordinal_position": 1},
                            {"column_name": "PATIENT", "data_type": "character varying", "ordinal_position": 2},
                            {"column_name": "ENCOUNTER", "data_type": "character varying", "ordinal_position": 3},
                            {"column_name": "CODE", "data_type": "bigint", "ordinal_position": 4},
                            {"column_name": "DESCRIPTION", "data_type": "character varying", "ordinal_position": 5},
                            {"column_name": "BASE_COST", "data_type": "numeric", "ordinal_position": 6},
                            {"column_name": "REASONCODE", "data_type": "bigint", "ordinal_position": 7},
                            {"column_name": "REASONDESCRIPTION", "data_type": "character varying", "ordinal_position": 8}
                        ]
                    },
                    "providers": {
                        "columns": [
                            {"column_name": "ID", "data_type": "character varying", "ordinal_position": 1},
                            {"column_name": "ORGANIZATION", "data_type": "character varying", "ordinal_position": 2},
                            {"column_name": "NAME", "data_type": "text", "ordinal_position": 3},
                            {"column_name": "GENDER", "data_type": "character", "ordinal_position": 4},
                            {"column_name": "SPECIALITY", "data_type": "text", "ordinal_position": 5},
                            {"column_name": "ADDRESS", "data_type": "character varying", "ordinal_position": 6},
                            {"column_name": "CITY", "data_type": "text", "ordinal_position": 7},
                            {"column_name": "STATE", "data_type": "text", "ordinal_position": 8},
                            {"column_name": "ZIP", "data_type": "character varying", "ordinal_position": 9},
                            {"column_name": "LAT", "data_type": "numeric", "ordinal_position": 10},
                            {"column_name": "LON", "data_type": "numeric", "ordinal_position": 11}
                        ]
                    }
                }
            }
        }
    }
    
    # Table descriptions with complete information
    table_descriptions = {
        "patients": {
            "description": "This table holds the list of patients' details with personal information like SSN, ID, name, birthdate etc.",
            "columns": {
                "ID": "unique patient identifier",
                "BIRTHDATE": "patient's birth date", 
                "DEATHDATE": "patient's death date",
                "SSN": "Social Security Number",
                "DRIVERS": "driver's license number",
                "PASSPORT": "passport number", 
                "PREFIX": "name prefix: Mr./Mrs./Miss",
                "FIRST": "first name",
                "LAST": "last name",
                "SUFFIX": "name suffix/degree",
                "MAIDEN": "maiden name",
                "MARTIAL": "marital status: M/S",
                "RACE": "race",
                "ETHINICITY": "ethnicity", 
                "GENDER": "sex: M/F",
                "BIRTHPLACE": "place of birth",
                "ADDRESS": "current address",
                "CITY": "city",
                "STATE": "state", 
                "COUNTY": "county",
                "ZIP": "postal code",
                "LAT": "latitude",
                "LON": "longitude",
                "HEALTHCARE_EXPENSES": "total expenses",
                "HEALTHCARE_COVERAGE": "total coverages"
            },
            "identification_fields": ["ID", "SSN", "DRIVERS", "PASSPORT"],
            "date_fields": ["BIRTHDATE", "DEATHDATE"],
            "relations": [
                "patients.ID ↔ allergies.PATIENT",
                "patients.ID ↔ encounters.PATIENT", 
                "patients.ID ↔ devices.PATIENT",
                "patients.ID ↔ conditions.PATIENT",
                "patients.ID ↔ medications.PATIENT",
                "patients.ID ↔ imaging_studies.PATIENT",
                "patients.ID ↔ immunizations.PATIENT",
                "patients.ID ↔ procedures.PATIENT",
                "patients.ID ↔ observations.PATIENT",
                "patients.ID ↔ payer_transitions.PATIENT",
                "patients.ID ↔ careplans.PATIENT"
            ]
        },
        "encounters": {
            "description": "This table holds the details of the patient's diagnosis (SNOMED Code) with its description along with the encounter class.",
            "columns": {
                "ID": "unique encounter identifier",
                "START": "start date of encounter",
                "STOP": "stop date of encounter", 
                "PATIENT": "FK → patients.ID",
                "ORGANIZATION": "FK → organizations.ID",
                "PROVIDER": "FK → providers.ID",
                "PAYER": "FK → payers.ID",
                "ENCOUNTERCLASS": "admission class",
                "CODE": "SNOMED code for reason of visit",
                "DESCRIPTION": "description of SNOMED code",
                "BASE_ENCOUNTER_COST": "base cost",
                "TOTAL_CLAIM_COST": "total claim cost",
                "PAYER_COVERAGE": "coverage amount",
                "REASONCODE": "SNOMED code for diagnosis",
                "REASONDESCRIPTION": "description of diagnosis code"
            },
            "identification_fields": ["ID", "PATIENT", "CODE", "REASONCODE"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "encounters.ID ↔ allergies.ENCOUNTER",
                "encounters.ID ↔ conditions.ENCOUNTER",
                "encounters.ID ↔ devices.ENCOUNTER", 
                "encounters.ID ↔ imaging_studies.ENCOUNTER",
                "encounters.ID ↔ immunizations.ENCOUNTER",
                "encounters.ID ↔ medications.ENCOUNTER",
                "encounters.ID ↔ careplans.ENCOUNTER",
                "encounters.ID ↔ observations.ENCOUNTER",
                "encounters.ID ↔ procedures.ENCOUNTER"
            ]
        },
        "allergies": {
            "description": "This table holds a list of allergies against the patients.",
            "columns": {
                "START": "date allergy began",
                "STOP": "date allergy ended",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID", 
                "CODE": "allergy code",
                "DESCRIPTION": "allergy description"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT", "CODE"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "patients.ID ↔ allergies.PATIENT",
                "encounters.ID ↔ allergies.ENCOUNTER"
            ]
        },
        "careplans": {
            "description": "This table holds a list of records for patient's diagnosis handled by nursing care.",
            "columns": {
                "ID": "unique careplan identifier",
                "START": "start date of careplan",
                "STOP": "stop date of careplan",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "SNOMED code",
                "DESCRIPTION": "description of code",
                "REASONCODE": "SNOMED code for reason",
                "REASONDESCRIPTION": "description of reason code"
            },
            "identification_fields": ["ID", "ENCOUNTER", "PATIENT", "CODE"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "encounters.ID ↔ careplans.ENCOUNTER",
                "patients.ID ↔ careplans.PATIENT"
            ]
        },
        "conditions": {
            "description": "This table holds a list of records for patient's conditions (SNOMED Code).",
            "columns": {
                "START": "start date of condition",
                "STOP": "stop date of condition",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "SNOMED code", 
                "DESCRIPTION": "description of code"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT", "CODE"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "encounters.ID ↔ conditions.ENCOUNTER",
                "patients.ID ↔ conditions.PATIENT"
            ]
        },
        "devices": {
            "description": "This table holds the list of patients who underwent device implantation.",
            "columns": {
                "START": "implant start date",
                "STOP": "implant stop date",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "SNOMED code",
                "DESCRIPTION": "description of code",
                "UDI": "unique device identifier"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT", "CODE"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "encounters.ID ↔ devices.ENCOUNTER",
                "patients.ID ↔ devices.PATIENT"
            ]
        },
        "imaging_studies": {
            "description": "This table holds the list of radiology study records for patients.",
            "columns": {
                "ID": "unique study identifier",
                "DATE": "study date",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "BODYSITE_CODE": "body site code",
                "BODYSITE_DESCRIPTION": "body site description",
                "MODALITY_CODE": "modality code",
                "MODALITY_DESCRIPTION": "modality description",
                "SOP_CODE": "DICOM SOP Class UID",
                "SOP_DESCRIPTION": "SOP description"
            },
            "identification_fields": ["ID", "ENCOUNTER", "PATIENT"],
            "date_fields": ["DATE"],
            "relations": [
                "encounters.ID ↔ imaging_studies.ENCOUNTER",
                "patients.ID ↔ imaging_studies.PATIENT"
            ]
        },
        "immunizations": {
            "description": "This table holds the list of patient vaccinations.",
            "columns": {
                "DATE": "vaccination date",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "CVX code",
                "DESCRIPTION": "vaccine description",
                "BASE_COST": "base cost"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT"],
            "date_fields": ["DATE"],
            "relations": [
                "encounters.ID ↔ immunizations.ENCOUNTER",
                "patients.ID ↔ immunizations.PATIENT"
            ]
        },
        "medications": {
            "description": "This table holds the list of patient medications with cost and coverage.",
            "columns": {
                "START": "medication start date",
                "STOP": "medication stop date",
                "PATIENT": "FK → patients.ID",
                "PAYER": "FK → payers.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "drug code",
                "DESCRIPTION": "drug description",
                "BASE_COST": "base cost",
                "PAYER_COVERAGE": "coverage amount",
                "DISPENSES": "quantity dispensed",
                "TOTALCOST": "total cost",
                "REASONCODE": "diagnosis code",
                "REASONDESCRIPTION": "diagnosis description"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT"],
            "date_fields": ["START", "STOP"],
            "relations": [
                "encounters.ID ↔ medications.ENCOUNTER",
                "patients.ID ↔ medications.PATIENT",
                "payers.ID ↔ medications.PAYER"
            ]
        },
        "observations": {
            "description": "This table holds patient vitals and observation readings.",
            "columns": {
                "DATE": "observation date",
                "PATIENT": "FK → patients.ID",
                "ENCOUTER": "FK → encounters.ID",
                "CODE": "LOINC code",
                "DESCRIPTION": "observation description",
                "VALUE": "reading value",
                "UNITS": "measurement units",
                "TYPE": "data type"
            },
            "identification_fields": ["ENCOUTER", "PATIENT"],
            "date_fields": ["DATE"],
            "relations": [
                "encounters.ID ↔ observations.ENCOUTER",
                "patients.ID ↔ observations.PATIENT"
            ]
        },
        "organizations": {
            "description": "This table holds organization details like name, address, and revenue.",
            "columns": {
                "ID": "unique organization identifier",
                "NAME": "organization name",
                "ADDRESS": "street address",
                "CITY": "city",
                "STATE": "state",
                "ZIP": "postal code",
                "LAT": "latitude",
                "LON": "longitude",
                "PHONE": "contact number",
                "REVENUE": "revenue amount"
            },
            "identification_fields": ["ID"],
            "date_fields": [],
            "relations": [
                "organizations.ID ↔ encounters.ORGANIZATION",
                "organizations.ID ↔ providers.ORGANIZATION"
            ]
        },
        "providers": {
            "description": "This table holds provider details including name, specialty, and location.",
            "columns": {
                "ID": "unique provider identifier",
                "ORGANIZATION": "FK → organizations.ID",
                "NAME": "provider name",
                "GENDER": "provider gender",
                "SPECIALITY": "clinical specialty",
                "ADDRESS": "street address",
                "CITY": "city",
                "STATE": "state",
                "ZIP": "postal code",
                "LAT": "latitude",
                "LON": "longitude"
            },
            "identification_fields": ["ID"],
            "date_fields": [],
            "relations": [
                "providers.ID ↔ encounters.PROVIDER",
                "organizations.ID ↔ providers.ORGANIZATION"
            ]
        },
        "procedures": {
            "description": "This table holds the list of procedures performed for patients.",
            "columns": {
                "DATE": "procedure date",
                "PATIENT": "FK → patients.ID",
                "ENCOUNTER": "FK → encounters.ID",
                "CODE": "SNOMED code",
                "DESCRIPTION": "procedure description",
                "BASE_COST": "base cost",
                "REASONCODE": "reason code",
                "REASONDESCRIPTION": "reason description"
            },
            "identification_fields": ["ENCOUNTER", "PATIENT"],
            "date_fields": ["DATE"],
            "relations": [
                "encounters.ID ↔ procedures.ENCOUNTER",
                "patients.ID ↔ procedures.PATIENT"
            ]
        },
        "payer_transitions": {
            "description": "This table holds insurance enrollment records for patients.",
            "columns": {
                "PATIENT": "FK → patients.ID",
                "START_YEAR": "insurance start year",
                "END_YEAR": "insurance end year",
                "PAYER": "FK → payers.ID",
                "OWNERSHIP": "policy ownership"
            },
            "identification_fields": ["PAYER", "PATIENT"],
            "date_fields": ["START_YEAR", "END_YEAR"],
            "relations": [
                "payers.ID ↔ payer_transitions.PAYER",
                "patients.ID ↔ payer_transitions.PATIENT"
            ]
        },
        "payers": {
            "description": "This table holds payer (insurance) details and coverage metrics.",
            "columns": {
                "ID": "unique payer identifier",
                "NAME": "payer name",
                "ADDRESS": "street address",
                "CITY": "city",
                "STATE_HEADQUARTERED": "headquarters state",
                "ZIP": "postal code",
                "PHONE": "contact number",
                "AMOUNT_COVERED": "covered amount",
                "AMOUNT_UNCOVERED": "uncovered amount",
                "REVENUE": "revenue",
                "COVERED_ENCOUTERS": "covered encounter count",
                "UNCOVERED_ENCOUNTERS": "uncovered encounter count",
                "COVERED_MEDICATIONS": "covered medication count",
                "UNCOVERED_MEDICATIONS": "uncovered medication count",
                "COVERED_PROCEDURES": "covered procedure count",
                "UNCOVERED_PROCEDURES": "uncovered procedure count",
                "COVERED_IMMUNIZATIONS": "covered immunization count",
                "UNCOVERED_IMMUNIZATIONS": "uncovered immunization count",
                "UNIQUE_CUSTOMERS": "unique customer count"
            },
            "identification_fields": ["ID"],
            "date_fields": [],
            "relations": [
                "payers.ID ↔ payer_transitions.PAYER",
                "payers.ID ↔ encounters.PAYER",
                "payers.ID ↔ medications.PAYER"
            ]
        }
    }
    
    formatted_schemas = []
    
    # Process each table from the JSON
    for schema_name, schema_data in json_data["schemas"].items():
        for table_name, table_data in schema_data["tables"].items():
            
            # Get table info
            table_info = table_descriptions.get(table_name, {})
            columns = table_data["columns"]
            column_count = len(columns)
            
            # Build formatted schema
            formatted_schema = {
                "id": f"{table_name}_full",
                "metadata": {
                    "table_name": table_name,
                    "section": "full",
                    "columns_count": column_count,
                    "relations_count": len(table_info.get("relations", []))
                },
                "text": f"Table: {table_name.title()}  \n{table_info.get('description', f'This table holds {table_name} related data.')}\nColumns:\n"
            }
            
            # Add column descriptions
            for column in columns:
                col_name = column["column_name"]
                col_description = table_info.get("columns", {}).get(col_name, "column description")
                formatted_schema["text"] += f"– {col_name} ({col_description})  \n"
            
            # Add identification fields
            id_fields = table_info.get("identification_fields", [])
            if id_fields:
                formatted_schema["text"] += f"Identification fields:\n{', '.join(id_fields)}\n"
            
            # Add date fields  
            date_fields = table_info.get("date_fields", [])
            if date_fields:
                formatted_schema["text"] += f"Date fields:\n{', '.join(date_fields)}\n"
            
            # Add relations
            relations = table_info.get("relations", [])
            if relations:
                formatted_schema["text"] += f"Relation fields:\n"
                for relation in relations:
                    formatted_schema["text"] += f"{relation}  \n"
            
            formatted_schemas.append(formatted_schema)
    
    return formatted_schemas

def save_formatted_schemas(schemas: List[Dict], filename: str = "complete_formatted_schema.json"):
    """Save formatted schemas to JSON file."""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(schemas, f, indent=2, ensure_ascii=False)
    print(f"Complete formatted schemas saved to {filename}")

def print_sample_schemas(schemas: List[Dict], count: int = 2):
    """Print sample schemas for preview."""
    print(f"Generated {len(schemas)} table schemas. Here are the first {count}:")
    print("=" * 80)
    
    for i, schema in enumerate(schemas[:count]):
        print(f"\nSchema {i+1}: {schema['id']}")
        print("-" * 40)
        print(f"Metadata: {schema['metadata']}")
        print(f"Text Preview:\n{schema['text'][:500]}...")
        print("-" * 40)

def main():
    """Main function to generate and save complete formatted schemas."""
    print("Generating complete formatted schemas from database extraction...")
    
    # Generate all formatted schemas
    formatted_schemas = generate_complete_formatted_schema()
    
    # Print sample schemas
    print_sample_schemas(formatted_schemas, 2)
    
    # Save to file
    save_formatted_schemas(formatted_schemas)
    
    # Print summary
    print(f"\nSummary:")
    print(f"- Total tables processed: {len(formatted_schemas)}")
    print(f"- Tables: {[schema['metadata']['table_name'] for schema in formatted_schemas]}")
    
    return formatted_schemas

if __name__ == "__main__":
    schemas = main()

Generating complete formatted schemas from database extraction...
Generated 15 table schemas. Here are the first 2:

Schema 1: allergies_full
----------------------------------------
Metadata: {'table_name': 'allergies', 'section': 'full', 'columns_count': 6, 'relations_count': 2}
Text Preview:
Table: Allergies  
This table holds a list of allergies against the patients.
Columns:
– START (date allergy began)  
– STOP (date allergy ended)  
– PATIENT (FK → patients.ID)  
– ENCOUNTER (FK → encounters.ID)  
– CODE (allergy code)  
– DESCRIPTION (allergy description)  
Identification fields:
ENCOUNTER, PATIENT, CODE
Date fields:
START, STOP
Relation fields:
patients.ID ↔ allergies.PATIENT  
encounters.ID ↔ allergies.ENCOUNTER  
...
----------------------------------------

Schema 2: careplans_full
----------------------------------------
Metadata: {'table_name': 'careplans', 'section': 'full', 'columns_count': 9, 'relations_count': 2}
Text Preview:
Table: Careplans  
This table holds a lis

In [64]:
import os
import psycopg2
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

# Get all tables in public schema
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE';
""")
tables = cur.fetchall()

# For each table, get column names
for table in tables:
    table_name = table[0]
    print(f"Table: {table_name}")
    cur.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = %s;
    """, (table_name,))
    columns = cur.fetchall()
    column_names = [col[0] for col in columns]
    print("  Columns:", ', '.join(column_names))
    print()

cur.close()
conn.close()


Table: allergies
  Columns: START, STOP, PATIENT, ENCOUNTER, CODE, DESCRIPTION

Table: procedures
  Columns: DATE, PATIENT, ENCOUNTER, CODE, DESCRIPTION, BASE_COST, REASONCODE, REASONDESCRIPTION

Table: providers
  Columns: ID, ORGANIZATION, NAME, GENDER, SPECIALITY, ADDRESS, CITY, STATE, ZIP, LAT, LON

Table: imaging_studies
  Columns: ID, DATE, PATIENT, ENCOUNTER, BODYSITE_CODE, BODYSITE_DESCRIPTION, MODALITY_CODE, MODALITY_DESCRIPTION, SOP_CODE, SOP_DESCRIPTION

Table: immunizations
  Columns: DATE, PATIENT, ENCOUNTER, CODE, DESCRIPTION, BASE_COST

Table: medications
  Columns: START, STOP, PATIENT, PAYER, ENCOUNTER, CODE, DESCRIPTION, BASE_COST, PAYER_COVERAGE, DISPENSES, TOTALCOST, REASONCODE, REASONDESCRIPTION

Table: observations
  Columns: DATE, PATIENT, ENCOUTER, CODE, DESCRIPTION, VALUE, UNITS, TYPE

Table: organizations
  Columns: ID, NAME, ADDRESS, CITY, STATE, ZIP, LAT, LON, PHONE, REVENUE

Table: patients
  Columns: PATIENT_ID, BIRTHDATE, DEATHDATE, SSN, DRIVERS, PASSPORT

In [59]:
import os
import psycopg2
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Get DB credentials from environment variables
db_params = {
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT',5432),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
}

# Table name (change as needed)
table_name = 'patients'

# Connect to PostgreSQL
conn = psycopg2.connect(**db_params)
cur = conn.cursor()

# Query to get column names
cur.execute(f"""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = %s
    ORDER BY ordinal_position;
""", (table_name,))

columns = [row[0] for row in cur.fetchall()]
print("Column names:", columns)

# Cleanup
cur.close()
conn.close()


Column names: ['PATIENT_ID', 'BIRTHDATE', 'DEATHDATE', 'SSN', 'DRIVERS', 'PASSPORT', 'PREFIX', 'FIRST', 'LAST', 'SUFFIX', 'MAIDEN', 'MARTIAL', 'RACE', 'ETHINICITY', 'GENDER', 'BIRTHPLACE', 'ADDRESS', 'CITY', 'STATE', 'COUNTY', 'ZIP', 'LAT', 'LON', 'HEALTHCARE_EXPENSES', 'HEALTHCARE_COVERAGE']


In [ ]:
import os
import psycopg2
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

# Get all tables in public schema
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE';
""")
tables = cur.fetchall()

# For each table, get column names
for table in tables:
    table_name = table[0]
    print(f"Table: {table_name}")
    cur.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = %s;
    """, (table_name,))
    columns = cur.fetchall()
    column_names = [col[0] for col in columns]
    print("  Columns:", ', '.join(column_names))
    print()

cur.close()
conn.close()


In [131]:
import os
import psycopg2
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

conn = psycopg2.connect(
    host=DB_HOST,
    database=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)

cur = conn.cursor()

# Get all tables in public schema
cur.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    AND table_type = 'BASE TABLE';
""")
tables = cur.fetchall()

# For each table, get column names
for table in tables:
    table_name = table[0]
    print(f"Table: {table_name}")
    cur.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'public' AND table_name = %s;
    """, (table_name,))
    columns = cur.fetchall()
    column_names = [col[0] for col in columns]
    print("  Columns:", ', '.join(column_names))
    print()

cur.close()
conn.close()


Table: allergies
  Columns: START, STOP, PATIENT_ID, ENCOUNTER_ID, ALLERGY_CODE, ALLERGY_DESCRIPTION

Table: procedures
  Columns: DATE, PATIENT_ID, ENCOUNTER_ID, PROCEDURE_CODE, PROCEDURE_DESCRIPTION, BASE_COST, REASONCODE, REASONDESCRIPTION

Table: providers
  Columns: PROVIDER_ID, ORGANIZATION_ID, NAME, GENDER, SPECIALITY, ADDRESS, CITY, STATE, ZIP, LAT, LON

Table: imaging_studies
  Columns: IMAGING_STUDIES_ID, DATE, PATIENT_ID, ENCOUNTER_ID, BODYSITE_CODE, BODYSITE_DESCRIPTION, MODALITY_CODE, MODALITY_DESCRIPTION, SOP_CODE, SOP_DESCRIPTION

Table: immunizations
  Columns: DATE, PATIENT_ID, ENCOUNTER_ID, IMMUNIZATION_CODE, IMMUNIZATION_DESCRIPTION, BASE_COST

Table: medications
  Columns: START, STOP, PATIENT_ID, PAYER_ID, ENCOUNTER_ID, MEDICATION_CODE, MEDICATION_DESCRIPTION, BASE_COST, PAYER_COVERAGE, DISPENSES, TOTALCOST, REASONCODE, REASONDESCRIPTION

Table: observations
  Columns: DATE, PATIENT_ID, ENCOUNTER_ID, OBSERVATION_CODE, OBSERVATION_DESCRIPTION, VALUE, UNITS, TYPE

Ta

In [93]:
import os
import psycopg2
from dotenv import load_dotenv

# Load env variables
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Connect to DB
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)
cur = conn.cursor()

# List of your tables
tables = [
    "allergies", "procedures", "providers", "imaging_studies", "immunizations",
    "medications", "observations", "organizations", "patients", "payer_transitions",
    "payers", "careplans", "conditions", "devices", "encounters"
]

# Function to check if 'ENCOUNTER' column exists and rename it
def rename_encounter_column(table):
    cur.execute("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = 'public'
        AND table_name = %s
        AND column_name = 'ENCOUNTER'
    """, (table,))
    if cur.fetchone():
        query = f'ALTER TABLE "public"."{table}" RENAME COLUMN "ENCOUNTER" TO "ENCOUNTER_ID";'
        print("Running:", query)
        cur.execute(query)
        conn.commit()
    else:
        print(f'Table "{table}" does not have column "ENCOUNTER". Skipping.')

if __name__ == "__main__":
    for table in tables:
        rename_encounter_column(table)

cur.close()
conn.close()
s

Table "allergies" does not have column "ENCOUNTER". Skipping.
Table "procedures" does not have column "ENCOUNTER". Skipping.
Table "providers" does not have column "ENCOUNTER". Skipping.
Table "imaging_studies" does not have column "ENCOUNTER". Skipping.
Table "immunizations" does not have column "ENCOUNTER". Skipping.
Table "medications" does not have column "ENCOUNTER". Skipping.
Table "observations" does not have column "ENCOUNTER". Skipping.
Table "organizations" does not have column "ENCOUNTER". Skipping.
Table "patients" does not have column "ENCOUNTER". Skipping.
Table "payer_transitions" does not have column "ENCOUNTER". Skipping.
Table "payers" does not have column "ENCOUNTER". Skipping.
Table "careplans" does not have column "ENCOUNTER". Skipping.
Table "conditions" does not have column "ENCOUNTER". Skipping.
Table "devices" does not have column "ENCOUNTER". Skipping.
Table "encounters" does not have column "ENCOUNTER". Skipping.


NameError: name 's' is not defined

In [2]:
import os
import psycopg2
from dotenv import load_dotenv

# Load env variables
load_dotenv()

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Connect to DB
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD
)
cur = conn.cursor()

# Change column names
# Example: ALTER TABLE old_table_name RENAME COLUMN old_column TO new_column;
def rename_columns(table, column_mapping):
    for old_col, new_col in column_mapping.items():
        query = f'ALTER TABLE "{table}" RENAME COLUMN "{old_col}" TO "{new_col}";'
        print("Running:", query)
        cur.execute(query)
    conn.commit()

# Update data in renamed columns
def update_data(table, updates, condition=None):
    set_clause = ", ".join([f'"{col}" = %s' for col in updates.keys()])
    values = list(updates.values())
    query = f'UPDATE "{table}" SET {set_clause}'
    if condition:
        query += f' WHERE {condition}'
    print("Running:", query)
    cur.execute(query, values)
    conn.commit()

# Example usage
if __name__ == "__main__":
    table = "conditions"
    column_mapping = {
   "PATIENT":"PATIENT_ID",
    }
    rename_columns(table, column_mapping)
    
    # Optional: Update data after renaming columns
   


Running: ALTER TABLE "conditions" RENAME COLUMN "PATIENT" TO "PATIENT_ID";
